In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

import src.features as F
import src.split as S
import src.metrics as M

df = pd.read_parquet("../data/processed/clean.parquet")
feats = F.build_features(df)
train, val, _ = S.time_split(feats)         # test is deliberately not bound to a name

y_tr, y_val = train["duration"], val["duration"]

# 1. constant: train median
pred_const = np.full(len(val), y_tr.median())

# 2. raw upstream estimate; NaN -> train median of the estimate
est_fill = train["estimated_total"].median()
pred_raw = val["estimated_total"].fillna(est_fill)

# 3. calibrated: duration ~ a + b * estimated_total, fit on train only
lin = LinearRegression().fit(train[["estimated_total"]].fillna(est_fill), y_tr)
pred_cal = lin.predict(val[["estimated_total"]].fillna(est_fill))

results = pd.DataFrame({
    "constant (train median)": M.evaluate(y_val, pred_const),
    "raw estimate": M.evaluate(y_val, pred_raw),
    "calibrated estimate": M.evaluate(y_val, pred_cal),
}).T.round(3)
display(results)
print(f"calibrated: duration = {lin.intercept_:.0f} + {lin.coef_[0]:.2f} * estimated_total")

,mae_min,rmse_min,within_10min,bias_min,over_pct
constant (train median),12.611,17.154,0.505,-3.190,0.483
raw estimate,32.916,36.674,0.026,-32.916,0.000
calibrated estimate,12.248,16.153,0.493,0.197,0.578


calibrated: duration = 1794 + 1.23 * estimated_total


In [2]:
import src.modeling as MD

train_m, (val_m,) = MD.add_store_features(train, [val])

ridge_rows = {}
for alpha in [1, 10, 100, 1000]:
    model = MD.build_ridge(alpha).fit(train_m, y_tr)
    ridge_rows[f"ridge alpha={alpha}"] = M.evaluate(y_val, model.predict(val_m))

results = pd.concat([results, pd.DataFrame(ridge_rows).T.round(3)])
display(results)

,mae_min,rmse_min,within_10min,bias_min,over_pct
constant (train median),12.611,17.154,0.505,-3.190,0.483
raw estimate,32.916,36.674,0.026,-32.916,0.000
calibrated estimate,12.248,16.153,0.493,0.197,0.578
ridge alpha=1,10.069,13.846,0.604,-0.258,0.563
ridge alpha=10,10.068,13.844,0.604,-0.258,0.563
ridge alpha=100,10.062,13.838,0.604,-0.257,0.563
ridge alpha=1000,10.046,13.827,0.605,-0.243,0.565


### Observations: Ridge (validation)

**1. Ridge clearly beats the calibrated baseline.** MAE 10.05 vs 12.25 min (-18%), RMSE 13.83 vs 16.15, and 60.5% of orders within +/-10 min vs 49.3%. Bias is near zero (-0.24 min), so it is not systematically early or late.

**2. Regularization strength is irrelevant here.** alpha from 1 to 1000 moves MAE by only 0.02 min. With ~138k rows and ~100 columns there is little to regularize, so I do not tune it further. **Decision:** alpha = 1000.

**3. Ridge sets the linear ceiling (~10.0 min).** It cannot learn interactions (supply x hour, store x time), which is what LightGBM is for.